In [20]:
# Some utitity libraries to test optuna
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Optuna is the library used for hyperparameter estimation based on many samplers.
# Mostly we use a sampler called TPE.
# Now this notebook is a complete walkthrough on why do we need optuna ? How did it come to existence ?
# Why other sampling methods failed ? 
# This notebook is not just about simply implement optuna -- It is about comprehending it's need.

In [ ]:
# Way before we had no such things as optimizer . All we had  was intutive guessing and manual looping over different hyperparameters.
# This was manual tuning .The code block would look like: 

In [11]:
# A simple iris dataset 
X,y = load_iris(return_X_y=True)

# Divide the data into three set of data
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
X_test,X_val,y_test,y_val = train_test_split(X_test,y_test,test_size=0.5,random_state=42)

In [ ]:
depths = [2,3,4,5,6,7,8]

best_score = -1
best_depth = None

for depth in depths:

    model = RandomForestClassifier(max_depth=depth)

    model.fit(X_train, y_train)

    score = model.score(X_val, y_val)

    print(f"Depth : {depth} | Score : {score}")

    if score > best_score:
        best_score = score
        best_depth = depth

print(best_depth)

# NOTE : This is codeblock is not to be evaluated but instead it showcases the manual hardship we had to do get the perfect the parameter.
# The dataset is too simple and model can easily learn all the patterns.

In [ ]:
# This was just a few depths and also one parameter .
# Now imagine big data , 100 of parameters and wait we have not talked about the combinations of each parameter yet.
# Multiple nested loops , memory hell and infinite-like time --- This is what we had to deal with for many years.

# We could have guessed parameters but there is a limit to the intutive guessing.
# Our function is basically: 

# From its perspective, it is solving        :   f(hyperparameters) → validation score
# The entire optimisation problem is simply  :   Find the hyperparameters that maximise (or minimise) this unknown function.

In [32]:
X, y = load_iris(return_X_y=True)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# BLACK BOX: THE EVALUATION ALGORITHM
# ==========================================
def evaluate(algorithm,params, X_data, y_data):
    """
    Acts as the black-box abstraction: f(hyperparameters) -> average validation score.
    The training algorithm (Random Forest) stays completely isolated inside here.
    It has no idea it is being searched or optimized.
    Please Make sure to pass the algorithm object not the function.
    """

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X_data):
        X_train, X_val = X_data[train_idx], X_data[val_idx]
        y_train, y_val = y_data[train_idx], y_data[val_idx]
        
        model = algorithm(**params, random_state=42)
        model.fit(X_train, y_train)
        
        score = model.score(X_val, y_val)
        fold_scores.append(score)
        
    return np.mean(fold_scores)


# ==========================================
# THE SEARCH ALGORITHM (Grid Search)
# ==========================================

param_grid = {
    'n_estimators':[50,100,125],
    'max_depth': [3, 5, None]
}


num_n_est = len(param_grid['n_estimators'])
num_depths = len(param_grid['max_depth'])
total_experiments = num_n_est * num_depths
folds = 3

print("--- COMPUTATIONAL COST ANALYSIS ---")
print(f"Hyperparameter 1 (n_estimators): {num_n_est} values")
print(f"Hyperparameter 2 (max_depth): {num_depths} values")
print(f"Total Unique Configurations: {num_n_est} × {num_depths} = {total_experiments} combinations")
print(f"Total Model Fits Required: {total_experiments} configs × {folds} CV folds = {total_experiments * folds} total runs")
print("-----------------------------------\n")

best_score = 0
best_params = {}

print("--- RUNNING GRID SEARCH ALGORITHM ---")

for n_est in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:

        current_params = {'n_estimators': n_est, 'max_depth': depth}
        avg_cv_score = evaluate(RandomForestClassifier,current_params, X_train_val, y_train_val)
        print(f"Testing Config: {current_params} | Result (Avg CV Score): {avg_cv_score:.4f}")
        
        if avg_cv_score > best_score:
            best_score = avg_cv_score
            best_params = current_params

print("\n--- SEARCH COMPLETE ---")

print(f"Optimal Hyperparameters: {best_params} with Best Avg CV Score: {best_score:.4f}")


best_model = RandomForestClassifier(**best_params, random_state=42)
best_model.fit(X_train_val, y_train_val)

final_test_score = best_model.score(X_test, y_test)
print(f"Best Test Score: {final_test_score:.4f}")


--- COMPUTATIONAL COST ANALYSIS ---
Hyperparameter 1 (n_estimators): 3 values
Hyperparameter 2 (max_depth): 3 values
Total Unique Configurations: 3 × 3 = 9 combinations
Total Model Fits Required: 9 configs × 3 CV folds = 27 total runs
-----------------------------------

--- RUNNING GRID SEARCH ALGORITHM ---
Testing Config: {'n_estimators': 50, 'max_depth': 3} | Result (Avg CV Score): 0.9667
Testing Config: {'n_estimators': 50, 'max_depth': 5} | Result (Avg CV Score): 0.9500
Testing Config: {'n_estimators': 50, 'max_depth': None} | Result (Avg CV Score): 0.9500
Testing Config: {'n_estimators': 100, 'max_depth': 3} | Result (Avg CV Score): 0.9667
Testing Config: {'n_estimators': 100, 'max_depth': 5} | Result (Avg CV Score): 0.9583
Testing Config: {'n_estimators': 100, 'max_depth': None} | Result (Avg CV Score): 0.9583
Testing Config: {'n_estimators': 125, 'max_depth': 3} | Result (Avg CV Score): 0.9667
Testing Config: {'n_estimators': 125, 'max_depth': 5} | Result (Avg CV Score): 0.9583

In [34]:
# This is a simple hardcoded working of how gridsearch csv works behind the scenes.
# Look closely how we working a grid like pattern : 
#   First we go on with first parameters and keep on looping till all the nested loop combinations are finished ,
#    exiting and going on to another outer loop and so on.

In [33]:
# Actually every combinations is the cartesian product of the parameters.
# Also we have a tool that gives us the cartesian product of all the combinations 
# So a better can be : 

In [74]:
from itertools import product # This library gives us the cartesian product (every combination)

# For example : 
param_grid = {
    'n_estimators':[50,100,125],
    'max_depth': [3, 5, None]
}
for x in product(*param_grid.values()):
    print(x)


(50, 3)
(50, 5)
(50, None)
(100, 3)
(100, 5)
(100, None)
(125, 3)
(125, 5)
(125, None)


In [37]:
# This is exactly what we wanted to further optimize our gridsearch

In [ ]:
# Improvised version of our gridsearch : 
X, y = load_iris(return_X_y=True)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# BLACK BOX: THE EVALUATION ALGORITHM
# ==========================================
def evaluate(algorithm,params, X_data, y_data):
    """
    Acts as the black-box abstraction: f(hyperparameters) -> average validation score.
    The training algorithm (Random Forest) stays completely isolated inside here.
    It has no idea it is being searched or optimized.
    Please Make sure to pass the algorithm object not the function.
    """

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X_data):
        X_train, X_val = X_data[train_idx], X_data[val_idx]
        y_train, y_val = y_data[train_idx], y_data[val_idx]
        
        model = algorithm(**params, random_state=42)
        model.fit(X_train, y_train)
        
        score = model.score(X_val, y_val)
        fold_scores.append(score)
        
    return np.mean(fold_scores)


# ==========================================
# THE SEARCH ALGORITHM (Grid Search)
# ==========================================

param_grid = {
    'n_estimators':[50,100,125],
    'max_depth': [3, 5, None]
}


num_n_est = len(param_grid['n_estimators'])
num_depths = len(param_grid['max_depth'])
total_experiments = num_n_est * num_depths
folds = 3

print("--- COMPUTATIONAL COST ANALYSIS ---")
print(f"Hyperparameter 1 (n_estimators): {num_n_est} values")
print(f"Hyperparameter 2 (max_depth): {num_depths} values")
print(f"Total Unique Configurations: {num_n_est} × {num_depths} = {total_experiments} combinations")
print(f"Total Model Fits Required: {total_experiments} configs × {folds} CV folds = {total_experiments * folds} total runs")
print("-----------------------------------\n")

best_score = 0
best_params = {}

print("--- RUNNING GRID SEARCH ALGORITHM ---")

params = list(param_grid.keys())

for values in product(*param_grid.values()):
    
    current_params = dict(zip(params,values))
    avg_cv_score = evaluate(RandomForestClassifier,current_params, X_train_val, y_train_val)
    print(f"Testing Config: {current_params} | Result (Avg CV Score): {avg_cv_score:.4f}")
        
    if avg_cv_score > best_score:
            best_score = avg_cv_score
            best_params = current_params

print("\n--- SEARCH COMPLETE ---")

print(f"Optimal Hyperparameters: {best_params} with Best Avg CV Score: {best_score:.4f}")


best_model = RandomForestClassifier(**best_params, random_state=42)
best_model.fit(X_train_val, y_train_val)

final_test_score = best_model.score(X_test, y_test)
print(f"Best Test Score: {final_test_score:.4f}")


--- COMPUTATIONAL COST ANALYSIS ---
Hyperparameter 1 (n_estimators): 3 values
Hyperparameter 2 (max_depth): 3 values
Total Unique Configurations: 3 × 3 = 9 combinations
Total Model Fits Required: 9 configs × 3 CV folds = 27 total runs
-----------------------------------

--- RUNNING GRID SEARCH ALGORITHM ---
Testing Config: {'n_estimators': 50, 'max_depth': 3} | Result (Avg CV Score): 0.9667
Testing Config: {'n_estimators': 50, 'max_depth': 5} | Result (Avg CV Score): 0.9500
Testing Config: {'n_estimators': 50, 'max_depth': None} | Result (Avg CV Score): 0.9500
Testing Config: {'n_estimators': 100, 'max_depth': 3} | Result (Avg CV Score): 0.9667
Testing Config: {'n_estimators': 100, 'max_depth': 5} | Result (Avg CV Score): 0.9583
Testing Config: {'n_estimators': 100, 'max_depth': None} | Result (Avg CV Score): 0.9583
Testing Config: {'n_estimators': 125, 'max_depth': 3} | Result (Avg CV Score): 0.9667
Testing Config: {'n_estimators': 125, 'max_depth': 5} | Result (Avg CV Score): 0.9583

# The similarities and difference between ManualTuning and GridSeachCV
------------------------------
## The Similarity (The Logic)
Both methods do the exact same work behind the scenes:

   1. They take a dictionary of hyperparameters.
   2. They map out every single possible combination (the "grid").
   3. They run cross-validation loops for every combination.
   4. They pick the combination with the highest average validation score. [3, 4, 5, 6, 7] 

------------------------------
## The Differences (The Execution)

| Feature | Manual Tuning (Your Loop Code) | Scikit-Learn's GridSearchCV |
|---|---|---|
| Code Length | Requires 30+ lines of nested loops and index tracking. | Requires only 3 to 4 lines of clean code. |
| Error Proneness | Easy to accidentally leak data or mess up index slicing. | Robust and thoroughly tested to prevent bugs. |
| Speed (Parallelization) | Runs on a single CPU core sequentially (slow). | Can run on all CPU cores at once using n_jobs=-1 (very fast). |
| Refitting | You must manually write code to retrain the model on the full data at the end. | Automatically retrains the best model for you (refit=True). |

------------------------------


In [27]:
# NOTE :  We did not import the GridSearchCV of the scikit-learn , which is much more robust.

## The Structural Flaws of Grid Search

* Zero Memory: It treats every single experiment as a completely isolated event.
* Refusal to Learn: It completely forgets the results of Experiment #1 before running Experiment #2.
* Wasted Labor: It forces the computer to finish testing poor configurations (like bad tea brands) even when early results prove they are failures.
* Static Beliefs: It cannot use trends from successful runs (like seeing accuracy jump at a certain depth) to narrow its focus on the "sweet spot."

## The Pivot to Random Search

* The Random Alternative: Instead of a rigid, systematic walk across a grid, it randomly throws darts at the search space.
* Surprisingly Better: It sounds chaotic, but mathematically, random guessing frequently beats a structured grid.
* Geometry over Rigidity: It avoids the trap of wasting time on unimportant parameters, setting the stage for the next breakthrough in optimization history.




In [ ]:
# Random Search is pure madness
# It is just simply throwing random darts at our grid search space grid and only evaluating that subset only .
"""
Grid Search assumes

    Every hyperparameter deserves equal attention.

Reality says

    Some hyperparameters dominate the model, while others have only a minor effect.

Instead of searching every point in grid search upto a subsample limit.

Is RandomSearchActually better than GridSearchCV when only a small subset of parameters influence the model peformance ? YES !!!

"""



## Why Random Search Can Outperform Grid Search
Let's use a simple analogy to understand the intuition behind this concept.
## The Problem

Imagine you are searching for buried treasure on a 100 m × 100 m beach. However, the treasure is buried somewhere along a single, narrow line. This means only one specific coordinate (dimension) actually matters for finding it.

------------------------------
## Method 1: Grid Search
Grid Search walks the beach in neat, rigid rows.

      □  □  □  □  □  □
      □  □  □  □  □  □
      □  □  □  □  □  □
      □  □  □  □  □  □


* The Flaw: Because it repeats the same coordinates across different rows, it tests very few unique values along that one critical dimension.

------------------------------
## Method 2: Random Search
Random Search drops 100 random pins across the beach.

      •      •       •
      •       •
      •       •      •
            •       •


* The Advantage: Because the pins are placed randomly, every single point tests a completely unique value along the important coordinate.
* The Result: It covers the critical search space much more efficiently than a rigid grid.

------------------------------
## Summary
> This is the core intuition from the foundational paper on hyperparameter optimization: when only a few hyperparameters significantly impact model performance (low effective dimensionality), Random Search explores unique configurations faster and more effectively than Grid Search.
